首先需要安装pymilvus的模型拓展
```
pip install "pymilvus[model]"
```  
还需安装
```
pip install peft, FlagEmbedding
```

# 以sentence_transformer粗召回语义匹配为例  
还内置了BGE、GTE、Jina等模型调用

In [ ]:
from pymilvus import model

In [ ]:
# 初始化嵌入模型实例
sentence_trans_ef = model.dense.SentenceTransformerEmbeddingFunction(
    model_name="aaa", #sen_trans模型
    device="cuda:0" #'cpu' or 'cuda:0'
)
# 可直接准备原始文本
docs = [
    "Artificial intelligence was founded as an academic discipline in 1956.",
    "Alan Turing was the first person to conduct substantial research in AI.",
    "Born in Maida Vale, London, Turing was raised in southern England.",
]
# 可直接嵌入为List[np.array]
docs_embeddings = sentence_trans_ef.encode_documents(docs)
# 维度与向量shape
print("Dim:", sentence_trans_ef.dim, docs_embeddings[0].shape)

# 以cross-encoder reranker精召回语义匹配为例  
还内置了BGE、Jina等reranker模型调用

In [3]:
from pymilvus.model.reranker import CrossEncoderRerankFunction

In [ ]:
# 初始化reranker实例
ce_rf = CrossEncoderRerankFunction(
    model_name="aaa", # cross_encoder_reranker
    device="cuda:0" #'cpu' or 'cuda:0'
)
# 提取prompt与知识库内容
query = "What event in 1956 marked the official birth of artificial intelligence as a discipline?"

documents = [
    "In 1950, Alan Turing published his seminal paper, 'Computing Machinery and Intelligence,' proposing the Turing Test as a criterion of intelligence, a foundational concept in the philosophy and development of artificial intelligence.",
    "The Dartmouth Conference in 1956 is considered the birthplace of artificial intelligence as a field; here, John McCarthy and others coined the term 'artificial intelligence' and laid out its basic goals.",
    "In 1951, British mathematician and computer scientist Alan Turing also developed the first program designed to play chess, demonstrating an early example of AI in game strategy.",
    "The invention of the Logic Theorist by Allen Newell, Herbert A. Simon, and Cliff Shaw in 1955 marked the creation of the first true AI program, which was capable of solving logic problems, akin to proving mathematical theorems."
]
# 可直接输出k个List[Tuple(id, score, document)]
reranker_results = ce_rf(query=query, documents=documents, top_k=3)
# belike
for result in results:
    print(f"Index: {result.index}")
    print(f"Score: {result.score:.6f}")
    print(f"Text: {result.text}\n")

## 这些便捷模型不仅可以直接使用，还可以作为schema或hybrid_search的参数直接传入milvus  
* 在内置自动嵌入时，比DOCKER的HuggingFace TEI更加方便  
* 也可以手动调用后，将向量传入milvus，数据管理更加灵活